# 0. Problem
## 1789. Primary Department for Each Employee — Easy
If an employee belongs to one department, return it. If they belong to multiple departments, return the row marked `primary_flag = 'Y'`.

Official: https://leetcode.com/problems/primary-department-for-each-employee/

# 1. Setup

In [ ]:
import pandas as pd
employee_rows=[(1,1,"N"),(2,1,"Y"),(2,2,"N"),(3,3,"N"),(4,2,"N"),(4,3,"Y"),(4,4,"N")]
employee_pd=pd.DataFrame(employee_rows,columns=["employee_id","department_id","primary_flag"])
employee_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
employee_spark=spark.createDataFrame(employee_rows,["employee_id","department_id","primary_flag"])
employee_spark.createOrReplaceTempView("Employee")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT employee_id, department_id
FROM Employee
WHERE primary_flag='Y'
   OR employee_id IN (
       SELECT employee_id
       FROM Employee
       GROUP BY employee_id
       HAVING COUNT(*)=1
   )
ORDER BY employee_id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
department_count=employee_pd.groupby("employee_id")["department_id"].transform("size")
result_pd=(employee_pd.loc[employee_pd["primary_flag"].eq("Y")|department_count.eq(1),["employee_id","department_id"]].sort_values("employee_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
w=Window.partitionBy("employee_id")
result_spark=(employee_spark.withColumn("department_count",F.count("department_id").over(w)).filter((F.col("primary_flag")=="Y")|(F.col("department_count")==1)).select("employee_id","department_id").orderBy("employee_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| group-size condition | subquery + `HAVING COUNT(*)=1` | `.transform("size")` | Window `count()` |
| choose flagged row | `primary_flag='Y'` | boolean mask | `.filter()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Employee

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: employee_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: employee_spark